# ML-03 — Frame Your Lane as an ML Task

**FlyRank AI Internship · Machine Learning · Week 2**  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook maps my Week 1 research question to an ML/analysis task. It is a **framing exercise**, not a trained or validated prediction system. The data demonstration uses the anonymized starter CSV; results are observational and intended to support review decisions.

**My research question:** Among pages with enough observed search activity, which should an SEO/content team review first for refresh, expansion, protection, consolidation, or monitoring?

*Review this plan in your own words before submitting; code results must be executed in Kaggle.*

## 1. My lane as an ML task (type)

**Primary task type: Ranking / scoring.** I want to create an interpretable priority queue of content pages, not just a yes/no list. A page with a higher score would be a candidate for **human review first**, not an automatic instruction to edit it.

- **Decision improved:** Which pages should the SEO/content team spend limited review time on first?
- **Who uses it:** An SEO analyst or content editor.
- **Action:** Inspect a page's evidence and consider refreshing, expanding, protecting, consolidating, or monitoring it.
- **Proposed output:** Pseudonymized page ID, rank/priority score, readable reason code, review suggestion, and uncertainty note.
- **Cost of mistakes:** False positives waste editorial time or prompt harmful changes; false negatives leave genuine problems unnoticed.

A secondary *teaching* task is to examine a current decline indicator, but classification is not my primary action output.

## 2. Target or proxy

**Near-term teaching proxy:** `decline_proxy = 1` if `trend_direction == "down"`, otherwise `0` in the starter snapshot. The proxy shows **current observed decline**, not future performance or the benefit of a content refresh. It is derived from `trend_direction` (itself related to `trend_pct`), so **neither `trend_direction` nor `trend_pct` may be used as input features** in a model predicting this proxy. Similarly, derived product flags or pre-computed scores must not leak the label.

**Better future target (if the warehouse's time history supports it):** A clearly defined *later-window* adverse outcome, e.g. decline in sufficient-volume search performance over a future 30-day window after the feature observation period. That target would require explicit thresholds, observation windows, adequate history, and leakage-safe temporal validation. It is **not available as a verified future target in this single 90-day snapshot**.

**Unit / analysis grain:** One pseudonymized content page (`client_id` + `content_id`) per row in the starter CSV. This is not a daily record. I will use pages with at least 100 recent impressions as an initial, transparent sample for examining review candidates.

**What I am not claiming:** A declining page is not guaranteed to benefit from updating, and the model cannot automatically decide the right editorial action.

## 3. Success metric

**Main candidate-ranking metric: Precision@50**, comparing the top 50 ranked pages against an explicit observed proxy in a held-out evaluation. This matches the practical constraint that an editorial team can review only a limited number of pages. I would also report recall, the number of eligible pages, and performance by client/content group where sample size allows.

**Baseline first:** Compare any trained scoring model with a transparent hand-written ranking rule (for example, declining pages with measurable demand; stale, visible pages). An ML approach is worthwhile only if it improves a predeclared metric without unacceptable complexity or leakage.

**Validation plan:** When modeling, split by `client_id` to avoid the same client's pages in both train and evaluation sets; for a future-time outcome, also respect chronological order. Do not present in-sample scores as generalization. Human review of top-ranked pages is required before content changes.

**No model or Precision@50 is computed in this framing notebook.** That is a later assignment.

## 4. The unit of analysis, as a real dataframe

The following cells clone my **public GitHub starter repository** when running in Kaggle, load its starter CSV, show real page-level rows, and create an illustrative `decline_proxy` column. No private client names, domains, URLs, or raw proprietary data are used.

**Note on data interpretation:** The starter dataset is a single trailing-90-day snapshot; IDs are pseudonymous. A value in the `decline_proxy` column illustrates an existing observed label, **not a new predictive result**.

In [6]:
# Kaggle-compatible setup: locate or clone my FlyRank starter repository.
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/rahadul231/flyrank-ml-internship-starter.git"
REPO_NAME = "flyrank-ml-internship-starter"
DATA_RELATIVE = Path("data/raw/content_refresh_anonymized.csv")

candidates = [
    Path("/kaggle/working") / REPO_NAME,
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
repo_root = next((p for p in candidates if (p / DATA_RELATIVE).is_file()), None)

if repo_root is None and Path("/kaggle/working").is_dir():
    repo_root = Path("/kaggle/working") / REPO_NAME
    if not repo_root.exists():
        print("Cloning public starter repository...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)

if repo_root is None or not (repo_root / DATA_RELATIVE).is_file():
    raise FileNotFoundError(
        "Starter CSV not found. In Kaggle, enable Internet in notebook settings "
        "and rerun the setup."
    )

csv_path = repo_root / DATA_RELATIVE
df = pd.read_csv(csv_path)
required = ["client_id", "content_id", "impressions_90d", "trend_direction"]
missing = [x for x in required if x not in df.columns]
assert not missing, f"Required columns missing: {missing}"

print("Working repository:", repo_root)
print("Starter CSV loaded:", csv_path)
print(f"Dataset shape: {len(df):,} rows x {len(df.columns):,} columns")
print("Setup successful. Dataframe df is ready.")

Working repository: /kaggle/working/flyrank-ml-internship-starter
Starter CSV loaded: /kaggle/working/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Dataset shape: 30,000 rows x 44 columns
Setup successful. Dataframe df is ready.


In [7]:
# Demonstrate the REAL analysis grain and a current-window teaching proxy.
# No ML training or prediction is performed here.
frame = df.copy()
frame["impressions_90d"] = pd.to_numeric(frame["impressions_90d"], errors="coerce")
frame["decline_proxy"] = frame["trend_direction"].astype(str).str.lower().eq("down").astype(int)

eligible = frame.loc[frame["impressions_90d"] >= 100].copy()
shown_cols = [
    "client_id", "content_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update",
    "trend_direction", "decline_proxy"
]
shown_cols = [col for col in shown_cols if col in eligible.columns]

print("One row = one pseudonymized content page in this starter snapshot.")
display(eligible[shown_cols].head(8))

keys = ["client_id", "content_id"]
print(f"Total snapshot rows: {len(frame):,}")
print(f"Eligible pages (at least 100 impressions): {len(eligible):,}")
print(f"Eligible pages with current decline proxy = 1: {eligible['decline_proxy'].sum():,}")
print(f"Unique client/page keys among eligible rows: {eligible[keys].drop_duplicates().shape[0]:,}")
print("Caution: 'decline_proxy' describes current observation, not a forecast or refresh benefit.")

One row = one pseudonymized content page in this starter snapshot.


,client_id,content_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,trend_direction,decline_proxy
0,client_f369cb89fc,content_304f48230142,3803,29,0.76,10.6,20,down,1
1,client_4e07408562,content_a1fb4e703a9e,15320,7,0.05,20.3,25,down,1
2,client_7f2253d7e2,content_9aa793d4d895,12581,11,0.09,36.5,20,down,1
3,client_19581e27de,content_331d6c4de07b,11751,58,0.49,6.2,22,stable,0
4,client_3fdba35f04,content_d99b7a2d90ca,19140,24,0.13,44.0,14,down,1
5,client_f369cb89fc,content_d4084a4bc775,3970,1,0.03,8.5,20,down,1
7,client_19581e27de,content_a63219c6e95a,1724,1,0.06,21.2,22,stable,0
8,client_6208ef0f77,content_5e6c160719bc,32574,29,0.09,46.0,20,down,1


Total snapshot rows: 30,000
Eligible pages (at least 100 impressions): 22,006
Eligible pages with current decline proxy = 1: 13,152
Unique client/page keys among eligible rows: 22,006
Caution: 'decline_proxy' describes current observation, not a forecast or refresh benefit.


### How I interpret the dataframe

Each displayed row represents a **single anonymized page** with its recent search signals and an illustrative current decline proxy. The page ID is for joining/ranking, not a predictive input. A future model should use only permissible signals available **before** the target outcome window, and must avoid `trend_direction`, `trend_pct`, and any derived product flag that reveals the decline label.

The eligible-page count and decline-proxy count above are **computed from the starter CSV during this run**. They are candidate-review context, **not** evidence that updating those pages causes improvement. If I choose a future decline target later, it must be constructed from separate chronological observations.

## 5. Why ML might beat a fixed rule here

A fixed rule is a necessary baseline because it is cheap, transparent, and sometimes good enough. However, two pages can have similar impressions but different search positions, CTR, age, content type, and engagement patterns. A learned model **might** combine such interactions to prioritize reviews more effectively than one threshold. It should be tested against the fixed rule rather than assumed better.

My success claim would be limited to **better observed ranking precision on held-out data**, if demonstrated. I would not claim the model discovers Google's ranking algorithm or proves that a content update causes higher traffic. Human judgment remains responsible for the final content action.

**Core first, AI second:** I chose the business decision, unit, target limitations, and metric; AI assistance helped draft the code and clarify potential leakage risks. I will review actual results after execution.

## 6. Self-check (complete after running in Kaggle)

- [ ] I have reviewed the wording and agree with the proposed Lane 2 framing.
- [x] I named the **ranking/scoring** task, the decision-maker, and a concrete review action.
- [x] I distinguished a **current decline proxy** from an actual future observed outcome.
- [x] I named **Precision@50** and a transparent baseline before training.
- [x] The setup and dataframe cells ran successfully and show real anonymized rows.
- [x] I described where leakage can occur and why held-out evaluation matters.
- [ ] I saved this notebook **with outputs** in `work/notebooks/w02_ml_task_framing.ipynb` on GitHub.

**Submission:** After verifying the committed executed notebook, submit the public repository URL: `https://github.com/rahadul231/flyrank-ml-internship-starter`. The notebook link or Kaggle URL is not the deliverable.